# Packages import

In [44]:
import os
import json
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# Ceneo Scraper

0. Prepare selenium


1. Provide url of the product's opinions page

In [45]:
product_code = "98826051"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

Prepare 

In [ ]:
path_to_driver = "D:\\chromedriver_win32\\chromedriver.exe"
s = Service(path_to_driver)
driver = webdriver.Chrome(service=s)
driver.get(url)
driver.maximize_window()
driver.find_element(by="xpath",value="//*[@id="js_cookie-consent-general"]/div/div[2]/button[1]").click()

2. Send request to provided url

In [46]:
response = requests.get(url)
print(response.status_code)


200


3. Fetch product name

In [47]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [48]:
product_name = page_dom.select_one("h1").get_text()
print(product_name)
print(type(product_name))

EyeLove Basic bi-weekly 3 szt soczewki dwutygodniowe
<class 'str'>


4. Fetch all opinions from the webpage

In [49]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinions to extract required data

In [50]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recommendation > em').get_text() if opinion.select_one('span.user-post__author-recommendation > em')else None,
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text() for c in opinion.select('div.review-feature__item--negative')],
        'helpful': opinion.select_one('button.vote-yes > span').get_text().strip(),
        'unhelpful': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publish_date': opinion.select_one('span.user-post__published >  time:nth-child(1)').get("datetime").strip(),
        'purchase_date': opinion.select_one('span.user-post__published >  time:nth-child(2)').get("datetime").strip()if opinion.select_one('span.user-post__author-recommendation > em')else None,
        
    }
    all_opinions.append(single_opinion)

6. Check if there is next page with opinions

In [ ]:
driver.find_element (by="xpath",value='//*[@id="reviews"]/div/div[6]/button[2]')

In [51]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page += 1

8. Save acquired opinions

In [52]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [53]:
with open(f'./opinions/{product_code}.json',"w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii=False)